> **FES · Fonaments d'Enginyeria de Software amb Python**

# 09 · Excepcions i validació

Una dada incorrecta o un fitxer absent poden interrompre un programa.
Les excepcions permeten comunicar aquests errors i tractar-los on podem
prendre una decisió.

## Objectius d'aprenentatge

- Gestionar `ValueError` i `FileNotFoundError` amb excepcions concretes.
- Explicar `try`, `except`, `else` i `finally`.
- Validar lectures abans d'acceptar-les i generar errors amb `raise`.

Dedicació orientativa de teoria i demostracions de 60 minuts. Les entrades
són textos proporcionats, de manera que el notebook s'executa sense interacció.

## Conversió i flux d'execució

`float("avaria")` genera `ValueError`. El bloc `except` tracta aquest error.
`else` s'executa si el `try` acaba correctament. `finally` s'executa en els dos
camins i permet efectuar una acció de finalització comuna.

Prediu quines línies apareixeran per a cada text. Limita el `try` a les
operacions que poden fallar i captura el tipus d'excepció esperat.

In [1]:
for text in ["21.5", "avaria"]:
    try:
        temperatura = float(text)
    except ValueError:
        print("La temperatura ha de ser numèrica.")
    else:
        print(f"Lectura convertida: {temperatura}")
    finally:
        print("Intent acabat.")

Lectura convertida: 21.5
Intent acabat.
La temperatura ha de ser numèrica.
Intent acabat.


## Un contracte per a la temperatura

Convertir a `float` és un primer pas. El sensor **simulat del curs** admet
temperatures de −40 a 85 °C, amb els extrems inclosos.
`raise ValueError(...)` comunica que una dada incompleix aquest contracte.

`float` també accepta textos com `"nan"` i `"inf"`. `math.isfinite` retorna
`False` per a aquests valors i per a `-inf`, i permet rebutjar-los abans de
comparar amb els límits. La funció valida i qui la crida decideix què mostrar.

In [2]:
import math


def validar_temperatura(text):
    """Retorna una temperatura finita dins del rang simulat [-40, 85]."""
    temperatura = float(text)
    if not math.isfinite(temperatura):
        raise ValueError("La temperatura ha de ser finita.")
    if temperatura < -40 or temperatura > 85:
        raise ValueError("Temperatura fora del rang simulat [-40, 85].")
    return temperatura


for text in ["-40", "21.5", "85", "86", "nan", "avaria"]:
    try:
        lectura = validar_temperatura(text)
    except ValueError as error:
        print(f"Rebutjada {text!r}: {error}")
    else:
        print(f"Acceptada: {lectura} °C")

Acceptada: -40.0 °C
Acceptada: 21.5 °C
Acceptada: 85.0 °C
Rebutjada '86': Temperatura fora del rang simulat [-40, 85].
Rebutjada 'nan': La temperatura ha de ser finita.
Rebutjada 'avaria': could not convert string to float: 'avaria'


## Aplicar la mateixa idea a la humitat

La humitat relativa del model pot anar de 0 a 100 %. Les dues funcions tenen
contractes diferents perquè representen magnituds diferents. Cada funció
concentra la validació de la seva magnitud en un sol lloc.

Comprova els extrems i un valor just fora del rang. Explica per què convé
validar abans de guardar una lectura.

In [3]:
def validar_humitat(text):
    """Retorna una humitat finita dins [0, 100] o genera ValueError."""
    humitat = float(text)
    if not math.isfinite(humitat):
        raise ValueError("La humitat ha de ser finita.")
    if humitat < 0 or humitat > 100:
        raise ValueError("La humitat ha d'estar entre 0 i 100.")
    return humitat


for text in ["0", "100", "101"]:
    try:
        print(validar_humitat(text))
    except ValueError as error:
        print(error)

0.0
100.0
La humitat ha d'estar entre 0 i 100.


## Fitxers absents i continuïtat del monitoratge

`FileNotFoundError` indica que falta el fitxer demanat. Creem una carpeta
temporal buida perquè la demostració produeixi aquest error sempre.
`Path` construeix la ruta i `with` gestiona els recursos. Al bloc 10 veurem
amb detall la lectura i escriptura. La carpeta temporal es neteja automàticament.

El segon exemple integra la validació en un bucle de monitoratge. Cada
lectura es valida abans d'afegir-la a la llista de lectures acceptades.

In [8]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "lectures_absents.txt"
    try:
        with open(ruta, encoding="utf-8") as fitxer:
            contingut = fitxer.read()
    except FileNotFoundError:
        print("Encara no hi ha cap fitxer de lectures.")
    else:
        print(contingut)

Encara no hi ha cap fitxer de lectures.


In [10]:
lectures_acceptades = []
for text in ["20", "avaria", "21", "90"]:
    try:
        lectura = validar_temperatura(text)
    except ValueError as error:
        print(f"Lectura descartada: {error}")
    else:
        lectures_acceptades.append(lectura)

print(lectures_acceptades)  # [20.0, 21.0]

Lectura descartada: could not convert string to float: 'avaria'
Lectura descartada: Temperatura fora del rang simulat [-40, 85].
[20.0, 21.0]


## Pràctica i ampliació

Resol els quatre [exercicis](01_excepcions_exercicis.py) i comprova els resultats
esperats i els casos límit. Després revisa les
[solucions](01_excepcions_solucions.py).

Els tres primers cobreixen conversió, contractes i gestió d'un fitxer absent.
El quart és un repte opcional de filtratge per al projecte de sensors.
Les excepcions pròpies queden com a ampliació posterior.